In [ ]:
# from google.colab import files
# uploaded = files.upload()

In [ ]:
import xml.etree.ElementTree as ET

import torch
import torch.nn as nn
from torch.utils.data.dataset import Dataset

import torchvision
import torchvision.transforms.v2
from torchvision.ops import box_iou
from torchvision import tv_tensors
from torchvision.io import read_image

import math

from torch.utils.data.dataloader import DataLoader
from torch.optim.lr_scheduler import MultiStepLR

import argparse
import numpy as np
import yaml
import random
from tqdm import tqdm

import cv2
import random
from tqdm import tqdm

import matplotlib.pyplot as plt


In [ ]:
import os
os.makedirs('config', exist_ok=True)


In [ ]:
%%writefile config/voc.yaml
dataset_params:
  train_im_sets: ['data/VOC2007']
  test_im_sets: ['data/VOC2007-test']
  num_classes : 21
  im_size : 300

model_params:
  im_channels : 3
  aspect_ratios : [
    [ 1., 2., 0.5 ],
    [ 1., 2., 3., 0.5, .333 ],
    [ 1., 2., 3., 0.5, .333 ],
    [ 1., 2., 3., 0.5, .333 ],
    [ 1., 2., 0.5 ],
    [ 1., 2., 0.5 ]
  ]
  scales : [0.1, 0.2, 0.375, 0.55, 0.725, 0.9]
  iou_threshold : 0.5
  low_score_threshold : 0.01
  neg_pos_ratio : 3
  pre_nms_topK : 400
  detections_per_img : 200
  nms_threshold : 0.45

train_params:
  task_name: 'voc'
  seed: 1111
  acc_steps: 1
  num_epochs: 100
  batch_size: 8
  lr_steps: [ 40, 50, 60, 70, 80, 90 ]
  lr: 0.001
  log_steps : 100
  infer_conf_threshold : 0.5
  ckpt_name: 'ssd_voc2007.pth'


In [ ]:
import os
import tarfile
import urllib.request

os.makedirs('data', exist_ok=True)

VOC_URLS = {
    'VOCtrainval_06-Nov-2007.tar': 'http://host.robots.ox.ac.uk/pascal/VOC/voc2007/VOCtrainval_06-Nov-2007.tar',
    'VOCtest_06-Nov-2007.tar': 'http://host.robots.ox.ac.uk/pascal/VOC/voc2007/VOCtest_06-Nov-2007.tar',
}

VOC_MIRROR_URLS = {
    'VOCtrainval_06-Nov-2007.tar': 'https://pjreddie.com/media/files/VOCtrainval_06-Nov-2007.tar',
    'VOCtest_06-Nov-2007.tar': 'https://pjreddie.com/media/files/VOCtest_06-Nov-2007.tar',
}


def download_and_extract(fname, url, mirror_url):
    tar_path = os.path.join('data', fname)
    if not os.path.exists(tar_path):
        print(f'Downloading {fname} ...')
        try:
            urllib.request.urlretrieve(url, tar_path)
        except Exception as e:
            print(f'Primary host failed ({e}), trying mirror...')
            urllib.request.urlretrieve(mirror_url, tar_path)
    else:
        print(f'{fname} already downloaded, skipping download')

    print(f'Extracting {fname} ...')
    with tarfile.open(tar_path) as tar:
        tar.extractall('data')


for fname, url in VOC_URLS.items():
    download_and_extract(fname, url, VOC_MIRROR_URLS[fname])

import shutil

if os.path.exists('data/VOCdevkit'):
    shutil.rmtree('data/VOCdevkit')

with tarfile.open('data/VOCtrainval_06-Nov-2007.tar') as tar:
    tar.extractall('data/_trainval_extract')
with tarfile.open('data/VOCtest_06-Nov-2007.tar') as tar:
    tar.extractall('data/_test_extract')

if os.path.exists('data/VOC2007'):
    shutil.rmtree('data/VOC2007')
if os.path.exists('data/VOC2007-test'):
    shutil.rmtree('data/VOC2007-test')

shutil.move('data/_trainval_extract/VOCdevkit/VOC2007', 'data/VOC2007')
shutil.move('data/_test_extract/VOCdevkit/VOC2007', 'data/VOC2007-test')

shutil.rmtree('data/_trainval_extract')
shutil.rmtree('data/_test_extract')

print('Dataset ready:')
print(' data/VOC2007      :', len(os.listdir('data/VOC2007/JPEGImages')), 'images')
print(' data/VOC2007-test :', len(os.listdir('data/VOC2007-test/JPEGImages')), 'images')


In [ ]:
def load_images_and_anns(im_sets, label2idx, ann_fname):
    im_infos = []

    for im_set in im_sets:
        im_names = []
        for line in open(os.path.join(
                im_set, 'ImageSets', 'Main', '{}.txt'.format(ann_fname))):
            im_names.append(line.strip())

        ann_dir = os.path.join(im_set, 'Annotations')
        im_dir = os.path.join(im_set, 'JPEGImages')
        for im_name in im_names:
            ann_file = os.path.join(ann_dir, '{}.xml'.format(im_name))
            im_info = {}
            ann_info = ET.parse(ann_file)
            root = ann_info.getroot()
            size = root.find('size')
            width = int(size.find('width').text)
            height = int(size.find('height').text)
            im_info['img_id'] = os.path.basename(ann_file).split('.xml')[0]
            im_info['filename'] = os.path.join(
                im_dir, '{}.jpg'.format(im_info['img_id'])
            )
            im_info['width'] = width
            im_info['height'] = height
            detections = []

            for obj in ann_info.findall('object'):
                det = {}
                label = label2idx[obj.find('name').text]
                difficult = int(obj.find('difficult').text)
                bbox_info = obj.find('bndbox')
                bbox = [
                    int(bbox_info.find('xmin').text) - 1,
                    int(bbox_info.find('ymin').text) - 1,
                    int(bbox_info.find('xmax').text) - 1,
                    int(bbox_info.find('ymax').text) - 1
                ]
                det['label'] = label
                det['bbox'] = bbox
                det['difficult'] = difficult
                detections.append(det)

            im_info['detections'] = detections
            im_infos.append(im_info)
    print('Total {} images found'.format(len(im_infos)))
    return im_infos


class VOCDataset(Dataset):
    def __init__(self, split, im_sets, im_size=300):
        self.split = split

        self.im_sets = im_sets
        self.fname = 'trainval' if self.split == 'train' else 'test'
        self.im_size = im_size
        self.im_mean = [123.0, 117.0, 104.0]
        self.imagenet_mean = [0.485, 0.456, 0.406]
        self.imagenet_std = [0.229, 0.224, 0.225]

        self.transforms = {
            'train': torchvision.transforms.v2.Compose([
                torchvision.transforms.v2.RandomPhotometricDistort(),
                torchvision.transforms.v2.RandomZoomOut(fill=self.im_mean),
                torchvision.transforms.v2.RandomIoUCrop(),
                torchvision.transforms.v2.RandomHorizontalFlip(p=0.5),
                torchvision.transforms.v2.Resize(size=(self.im_size, self.im_size)),
                torchvision.transforms.v2.SanitizeBoundingBoxes(
                    labels_getter=lambda transform_input:
                    (transform_input[1]["labels"], transform_input[1]["difficult"])),
                torchvision.transforms.v2.ToPureTensor(),
                torchvision.transforms.v2.ToDtype(torch.float32, scale=True),
                torchvision.transforms.v2.Normalize(mean=self.imagenet_mean,
                                                    std=self.imagenet_std)

            ]),
            'test': torchvision.transforms.v2.Compose([
                torchvision.transforms.v2.Resize(size=(self.im_size, self.im_size)),
                torchvision.transforms.v2.ToPureTensor(),
                torchvision.transforms.v2.ToDtype(torch.float32, scale=True),
                torchvision.transforms.v2.Normalize(mean=self.imagenet_mean,
                                                    std=self.imagenet_std)
            ]),
        }

        classes = [
            'person', 'bird', 'cat', 'cow', 'dog', 'horse', 'sheep',
            'aeroplane', 'bicycle', 'boat', 'bus', 'car', 'motorbike', 'train',
            'bottle', 'chair', 'diningtable', 'pottedplant', 'sofa', 'tvmonitor'
        ]
        classes = sorted(classes)
        classes = ['background'] + classes

        self.label2idx = {classes[idx]: idx for idx in range(len(classes))}
        self.idx2label = {idx: classes[idx] for idx in range(len(classes))}
        print(self.idx2label)
        self.images_info = load_images_and_anns(self.im_sets,
                                                self.label2idx,
                                                self.fname,
                                                )

    def __len__(self):
        return len(self.images_info)

    def __getitem__(self, index):
        im_info = self.images_info[index]
        im = read_image(im_info['filename'])

        targets = {}
        targets['bboxes'] = tv_tensors.BoundingBoxes(
            [detection['bbox'] for detection in im_info['detections']],
            format='XYXY', canvas_size=im.shape[-2:])
        targets['labels'] = torch.as_tensor(
            [detection['label'] for detection in im_info['detections']])
        targets['difficult'] = torch.as_tensor(
            [detection['difficult']for detection in im_info['detections']])

        transformed_info = self.transforms[self.split](im, targets)
        im_tensor, targets = transformed_info

        h, w = im_tensor.shape[-2:]
        wh_tensor = torch.as_tensor([[w, h, w, h]]).expand_as(targets['bboxes'])
        targets['bboxes'] = targets['bboxes'] / wh_tensor
        return im_tensor, targets, im_info['filename']

In [ ]:
def boxes_to_transformation_targets(ground_truth_boxes,
                                    default_boxes,
                                    weights=(10., 10., 5., 5.)):
    widths = default_boxes[:, 2] - default_boxes[:, 0]
    heights = default_boxes[:, 3] - default_boxes[:, 1]
    center_x = default_boxes[:, 0] + 0.5 * widths
    center_y = default_boxes[:, 1] + 0.5 * heights

    gt_widths = (ground_truth_boxes[:, 2] - ground_truth_boxes[:, 0])
    gt_heights = ground_truth_boxes[:, 3] - ground_truth_boxes[:, 1]
    gt_center_x = ground_truth_boxes[:, 0] + 0.5 * gt_widths
    gt_center_y = ground_truth_boxes[:, 1] + 0.5 * gt_heights

    targets_dx = weights[0] * (gt_center_x - center_x) / widths
    targets_dy = weights[1] * (gt_center_y - center_y) / heights
    targets_dw = weights[2] * torch.log(gt_widths / widths)
    targets_dh = weights[3] * torch.log(gt_heights / heights)
    regression_targets = torch.stack((targets_dx,
                                      targets_dy,
                                      targets_dw,
                                      targets_dh), dim=1)
    return regression_targets


def apply_regression_pred_to_default_boxes(box_transform_pred,
                                           default_boxes,
                                           weights=(10., 10., 5., 5.)):

    w = default_boxes[:, 2] - default_boxes[:, 0]
    h = default_boxes[:, 3] - default_boxes[:, 1]
    center_x = default_boxes[:, 0] + 0.5 * w
    center_y = default_boxes[:, 1] + 0.5 * h

    dx = box_transform_pred[..., 0] / weights[0]
    dy = box_transform_pred[..., 1] / weights[1]
    dw = box_transform_pred[..., 2] / weights[2]
    dh = box_transform_pred[..., 3] / weights[3]

    pred_center_x = dx * w + center_x
    pred_center_y = dy * h + center_y
    pred_w = torch.exp(dw) * w
    pred_h = torch.exp(dh) * h

    pred_box_x1 = pred_center_x - 0.5 * pred_w
    pred_box_y1 = pred_center_y - 0.5 * pred_h
    pred_box_x2 = pred_center_x + 0.5 * pred_w
    pred_box_y2 = pred_center_y + 0.5 * pred_h

    pred_boxes = torch.stack((
        pred_box_x1,
        pred_box_y1,
        pred_box_x2,
        pred_box_y2),
        dim=-1)
    return pred_boxes


def generate_default_boxes(feat, aspect_ratios, scales):
    default_boxes = []
    for k in range(len(feat)):
        s_prime_k = math.sqrt(scales[k] * scales[k + 1])
        wh_pairs = [[s_prime_k, s_prime_k]]

        for ar in aspect_ratios[k]:
            sq_ar = math.sqrt(ar)
            w = scales[k] * sq_ar
            h = scales[k] / sq_ar

            wh_pairs.extend([[w, h]])

        feat_h, feat_w = feat[k].shape[-2:]

        shifts_x = ((torch.arange(0, feat_w) + 0.5) / feat_w).to(torch.float32)
        shifts_y = ((torch.arange(0, feat_h) + 0.5) / feat_h).to(torch.float32)
        shift_y, shift_x = torch.meshgrid(shifts_y, shifts_x, indexing="ij")
        shift_x = shift_x.reshape(-1)
        shift_y = shift_y.reshape(-1)

        shifts = torch.stack((shift_x, shift_y) * len(wh_pairs), dim=-1).reshape(-1, 2)

        wh_pairs = torch.as_tensor(wh_pairs)

        wh_pairs = wh_pairs.repeat((feat_h * feat_w), 1)

        default_box = torch.cat((shifts, wh_pairs), dim=1)

        default_boxes.append(default_box)
    default_boxes = torch.cat(default_boxes, dim=0)
    dboxes = []
    for _ in range(feat[0].size(0)):
        dboxes_in_image = default_boxes
        dboxes_in_image = torch.cat(
            [
                (dboxes_in_image[:, :2] - 0.5 * dboxes_in_image[:, 2:]),
                (dboxes_in_image[:, :2] + 0.5 * dboxes_in_image[:, 2:]),
            ],
            -1,
        )
        dboxes.append(dboxes_in_image.to(feat[0].device))
    return dboxes


class SSD(nn.Module):
    def __init__(self, config, num_classes=21):
        super().__init__()
        self.aspect_ratios = config['aspect_ratios']

        self.scales = config['scales']
        self.scales.append(1.0)

        self.num_classes = num_classes
        self.iou_threshold = config['iou_threshold']
        self.low_score_threshold = config['low_score_threshold']
        self.neg_pos_ratio = config['neg_pos_ratio']
        self.pre_nms_topK = config['pre_nms_topK']
        self.nms_threshold = config['nms_threshold']
        self.detections_per_img = config['detections_per_img']

        backbone = torchvision.models.vgg16(
            weights=torchvision.models.VGG16_Weights.IMAGENET1K_V1
        )

        max_pool_pos = [idx for idx, layer in enumerate(list(backbone.features))
                        if isinstance(layer, nn.MaxPool2d)]
        max_pool_stage_3_pos = max_pool_pos[-3]  
        max_pool_stage_4_pos = max_pool_pos[-2]  

        backbone.features[max_pool_stage_3_pos].ceil_mode = True
        self.features = nn.Sequential(*backbone.features[:max_pool_stage_4_pos])
        self.scale_weight = nn.Parameter(torch.ones(512) * 20)

        fcs = nn.Sequential(
            nn.MaxPool2d(kernel_size=3, stride=1, padding=1),
            nn.Conv2d(in_channels=512, out_channels=1024, kernel_size=3,
                      padding=6, dilation=6),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels=1024, out_channels=1024, kernel_size=1),
            nn.ReLU(inplace=True),
        )
        self.conv5_3_fc = nn.Sequential(
            *backbone.features[max_pool_stage_4_pos:-1],
            fcs,
        )

        self.conv8_2 = nn.Sequential(
            nn.Conv2d(1024, 256, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 512, kernel_size=3, padding=1,
                      stride=2),
            nn.ReLU(inplace=True)
        )

        self.conv9_2 = nn.Sequential(
            nn.Conv2d(512, 128, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, kernel_size=3, padding=1,
                      stride=2),
            nn.ReLU(inplace=True)
        )

        self.conv10_2 = nn.Sequential(
            nn.Conv2d(256, 128, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, kernel_size=3),
            nn.ReLU(inplace=True)
        )

        self.conv11_2 = nn.Sequential(
            nn.Conv2d(256, 128, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, kernel_size=3),
            nn.ReLU(inplace=True)
        )

        out_channels = [512, 1024, 512, 256, 256, 256]

        self.cls_heads = nn.ModuleList()
        for channels, aspect_ratio in zip(out_channels, self.aspect_ratios):
            self.cls_heads.append(nn.Conv2d(channels,
                                            self.num_classes * (len(aspect_ratio)+1),
                                            kernel_size=3,
                                            padding=1))

        self.bbox_reg_heads = nn.ModuleList()
        for channels, aspect_ratio in zip(out_channels, self.aspect_ratios):
            self.bbox_reg_heads.append(nn.Conv2d(channels, 4 * (len(aspect_ratio)+1),
                                                 kernel_size=3,
                                                 padding=1))

        for layer in fcs.modules():
            if isinstance(layer, nn.Conv2d):
                torch.nn.init.xavier_uniform_(layer.weight)
                if layer.bias is not None:
                    torch.nn.init.constant_(layer.bias, 0.0)

        for conv_module in [self.conv8_2, self.conv9_2, self.conv10_2, self.conv11_2]:
            for layer in conv_module.modules():
                if isinstance(layer, nn.Conv2d):
                    torch.nn.init.xavier_uniform_(layer.weight)
                    if layer.bias is not None:
                        torch.nn.init.constant_(layer.bias, 0.0)

        for module in self.cls_heads:
            torch.nn.init.xavier_uniform_(module.weight)
            if module.bias is not None:
                torch.nn.init.constant_(module.bias, 0.0)
        for module in self.bbox_reg_heads:
            torch.nn.init.xavier_uniform_(module.weight)
            if module.bias is not None:
                torch.nn.init.constant_(module.bias, 0.0)

    def compute_loss(
            self,
            targets,
            cls_logits,
            bbox_regression,
            default_boxes,
            matched_idxs,
    ):
        num_foreground = 0
        bbox_loss = []
        cls_targets = []
        for (
            targets_per_image,
            bbox_regression_per_image,
            cls_logits_per_image,
            default_boxes_per_image,
            matched_idxs_per_image,
        ) in zip(targets, bbox_regression, cls_logits, default_boxes, matched_idxs):
            fg_idxs_per_image = torch.where(matched_idxs_per_image >= 0)[0]
            foreground_matched_idxs_per_image = matched_idxs_per_image[
                fg_idxs_per_image
            ]
            num_foreground += foreground_matched_idxs_per_image.numel()

            matched_gt_boxes_per_image = targets_per_image["boxes"][
                foreground_matched_idxs_per_image
            ]
            bbox_regression_per_image = bbox_regression_per_image[fg_idxs_per_image, :]
            default_boxes_per_image = default_boxes_per_image[fg_idxs_per_image, :]
            target_regression = boxes_to_transformation_targets(
                matched_gt_boxes_per_image,
                default_boxes_per_image)

            bbox_loss.append(
                torch.nn.functional.smooth_l1_loss(bbox_regression_per_image,
                                                   target_regression,
                                                   reduction='sum')
            )

            gt_classes_target = torch.zeros(
                (cls_logits_per_image.size(0),),
                dtype=targets_per_image["labels"].dtype,
                device=targets_per_image["labels"].device,
            )
            gt_classes_target[fg_idxs_per_image] = targets_per_image["labels"][
                foreground_matched_idxs_per_image
            ]
            cls_targets.append(gt_classes_target)

        bbox_loss = torch.stack(bbox_loss)
        cls_targets = torch.stack(cls_targets)  # (B, 8732)

        num_classes = cls_logits.size(-1)
        cls_loss = torch.nn.functional.cross_entropy(cls_logits.view(-1, num_classes),
                                                     cls_targets.view(-1),
                                                     reduction="none").view(
            cls_targets.size()
        )

        foreground_idxs = cls_targets > 0
        num_negative = self.neg_pos_ratio * foreground_idxs.sum(1, keepdim=True)

        negative_loss = cls_loss.clone()
        negative_loss[foreground_idxs] = -float("inf")
        values, idx = negative_loss.sort(1, descending=True)
        background_idxs = idx.sort(1)[1] < num_negative
        N = max(1, num_foreground)
        return {
            "bbox_regression": bbox_loss.sum() / N,
            "classification": (cls_loss[foreground_idxs].sum() +
                               cls_loss[background_idxs].sum()) / N,
        }

    def forward(self, x, targets=None):
        conv_4_3_out = self.features(x)

        conv_4_3_out_scaled = (self.scale_weight.view(1, -1, 1, 1) *
                               torch.nn.functional.normalize(conv_4_3_out))

        conv_5_3_fc_out = self.conv5_3_fc(conv_4_3_out)
        conv8_2_out = self.conv8_2(conv_5_3_fc_out)
        conv9_2_out = self.conv9_2(conv8_2_out)
        conv10_2_out = self.conv10_2(conv9_2_out)
        conv11_2_out = self.conv11_2(conv10_2_out)

        # Feature maps for predictions
        outputs = [
            conv_4_3_out_scaled,  # 38 x 38
            conv_5_3_fc_out,  # 19 x 19
            conv8_2_out,  # 10 x 10
            conv9_2_out,  # 5 x 5
            conv10_2_out,  # 3 x 3
            conv11_2_out,   # 1 x 1
        ]

        cls_logits = []
        bbox_reg_deltas = []
        for i, features in enumerate(outputs):
            cls_feat_i = self.cls_heads[i](features)
            bbox_reg_feat_i = self.bbox_reg_heads[i](features)

            # Cls output from (B, A * num_classes, H, W) to (B, HWA, num_classes).
            N, _, H, W = cls_feat_i.shape
            cls_feat_i = cls_feat_i.view(N, -1, self.num_classes, H, W)
            # (B, A, num_classes, H, W)
            cls_feat_i = cls_feat_i.permute(0, 3, 4, 1, 2)  # (B, H, W, A, num_classes)
            cls_feat_i = cls_feat_i.reshape(N, -1, self.num_classes)
            # (B, HWA, num_classes)
            cls_logits.append(cls_feat_i)

            N, _, H, W = bbox_reg_feat_i.shape
            bbox_reg_feat_i = bbox_reg_feat_i.view(N, -1, 4, H, W)  # (B, A, 4, H, W)
            bbox_reg_feat_i = bbox_reg_feat_i.permute(0, 3, 4, 1, 2)  # (B, H, W, A, 4)
            bbox_reg_feat_i = bbox_reg_feat_i.reshape(N, -1, 4)  # Size=(B, HWA, 4)
            bbox_reg_deltas.append(bbox_reg_feat_i)

        cls_logits = torch.cat(cls_logits, dim=1)  # (B, 8732, num_classes)
        bbox_reg_deltas = torch.cat(bbox_reg_deltas, dim=1)  # (B, 8732, 4)

        default_boxes = generate_default_boxes(outputs, self.aspect_ratios, self.scales)

        losses = {}
        detections = []
        if self.training:
            matched_idxs = []
            for default_boxes_per_image, targets_per_image in zip(default_boxes,
                                                                  targets):
                if targets_per_image["boxes"].numel() == 0:
                    matched_idxs.append(
                        torch.full(
                            (default_boxes_per_image.size(0),), -1,
                            dtype=torch.int64,
                            device=default_boxes_per_image.device
                        )
                    )
                    continue
                iou_matrix = box_iou(targets_per_image["boxes"],
                                     default_boxes_per_image)
                matched_vals, matches = iou_matrix.max(dim=0)

                below_low_threshold = matched_vals < self.iou_threshold
                matches[below_low_threshold] = -1

                _, highest_quality_pred_foreach_gt = iou_matrix.max(dim=1)
                matches[highest_quality_pred_foreach_gt] = torch.arange(
                    highest_quality_pred_foreach_gt.size(0), dtype=torch.int64,
                    device=highest_quality_pred_foreach_gt.device
                )
                matched_idxs.append(matches)
            losses = self.compute_loss(targets, cls_logits, bbox_reg_deltas,
                                       default_boxes, matched_idxs)
        else:
            # For test time we do the following:
            # 1. Convert default_boxes to boxes using predicted bbox regression deltas
            # 2. Low score filtering
            # 3. Pre-NMS TopK filtering
            # 4. NMS
            # 5. Post NMS TopK Filtering
            cls_scores = torch.nn.functional.softmax(cls_logits, dim=-1)
            num_classes = cls_scores.size(-1)

            for bbox_deltas_i, cls_scores_i, default_boxes_i in zip(bbox_reg_deltas,
                                                                    cls_scores,
                                                                    default_boxes):
                boxes = apply_regression_pred_to_default_boxes(bbox_deltas_i,
                                                               default_boxes_i)
                # Ensure all values are between 0-1
                boxes.clamp_(min=0., max=1.)

                pred_boxes = []
                pred_scores = []
                pred_labels = []
                for label in range(1, num_classes):
                    score = cls_scores_i[:, label]

                    # Remove low scoring boxes of this class
                    keep_idxs = score > self.low_score_threshold
                    score = score[keep_idxs]
                    box = boxes[keep_idxs]

                    # keep only topk scoring predictions of this class
                    score, top_k_idxs = score.topk(min(self.pre_nms_topK, len(score)))
                    box = box[top_k_idxs]

                    pred_boxes.append(box)
                    pred_scores.append(score)
                    pred_labels.append(torch.full_like(score, fill_value=label,
                                                       dtype=torch.int64,
                                                       device=cls_scores.device))

                pred_boxes = torch.cat(pred_boxes, dim=0)
                pred_scores = torch.cat(pred_scores, dim=0)
                pred_labels = torch.cat(pred_labels, dim=0)

                # Class wise NMS
                keep_mask = torch.zeros_like(pred_scores, dtype=torch.bool)
                for class_id in torch.unique(pred_labels):
                    curr_indices = torch.where(pred_labels == class_id)[0]
                    curr_keep_idxs = torch.ops.torchvision.nms(pred_boxes[curr_indices],
                                                               pred_scores[curr_indices],
                                                               self.nms_threshold)
                    keep_mask[curr_indices[curr_keep_idxs]] = True
                keep_indices = torch.where(keep_mask)[0]
                post_nms_keep_indices = keep_indices[pred_scores[keep_indices].sort(
                    descending=True)[1]]
                keep = post_nms_keep_indices[:self.detections_per_img]
                pred_boxes, pred_scores, pred_labels = (pred_boxes[keep],
                                                        pred_scores[keep],
                                                        pred_labels[keep])

                detections.append(
                    {
                        "boxes": pred_boxes,
                        "scores": pred_scores,
                        "labels": pred_labels,
                    }
                )
        return losses, detections

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# if torch.backends.mps.is_available():
#     device = torch.device('mps')
#     print('Using mps')


# def collate_function(data):
#     return tuple(zip(*data))


# def train(args):
#     # Read the config file
#     with open(args.config_path, 'r') as file:
#         try:
#             config = yaml.safe_load(file)
#         except yaml.YAMLError as exc:
#             print(exc)
#     print(config)

#     dataset_config = config['dataset_params']
#     train_config = config['train_params']

#     seed = train_config['seed']
#     torch.manual_seed(seed)
#     np.random.seed(seed)
#     random.seed(seed)
#     if device.type == 'cuda':
#         torch.cuda.manual_seed_all(seed)

#     voc = VOCDataset('train',
#                      im_sets=dataset_config['train_im_sets'],
#                      im_size=dataset_config['im_size'])
#     train_dataset = DataLoader(voc,
#                                batch_size=train_config['batch_size'],
#                                shuffle=True,
#                                collate_fn=collate_function)

#     model = SSD(config=config['model_params'],
#                 num_classes=dataset_config['num_classes'])
#     model.to(device)
#     model.train()

#     if not os.path.exists(train_config['task_name']):
#         os.mkdir(train_config['task_name'])

#     optimizer = torch.optim.SGD(lr=train_config['lr'],
#                                 params=model.parameters(),
#                                 weight_decay=5E-4, momentum=0.9)
#     lr_scheduler = MultiStepLR(optimizer, milestones=train_config['lr_steps'], gamma=0.5)
#     acc_steps = train_config['acc_steps']
#     num_epochs = train_config['num_epochs']

#     latest_ckpt_path = os.path.join(train_config['task_name'],
#                                      'latest_' + train_config['ckpt_name'])
#     start_epoch = 0
#     steps = 0

#     if os.path.exists(latest_ckpt_path):
#         print('Loading checkpoint as one exists')
#         checkpoint = torch.load(latest_ckpt_path, map_location=device)
#         model.load_state_dict(checkpoint['model_state_dict'])
#         optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
#         lr_scheduler.load_state_dict(checkpoint['lr_scheduler_state_dict'])
#         start_epoch = checkpoint['epoch'] + 1
#         steps = checkpoint.get('steps', 0)
#         print(f'Resuming from epoch {start_epoch}')

#     for i in range(start_epoch, num_epochs):
#         ssd_classification_losses = []
#         ssd_localization_losses = []
#         for idx, (ims, targets, _) in enumerate(tqdm(train_dataset)):
#             for target in targets:
#                 target['boxes'] = target['bboxes'].float().to(device)
#                 del target['bboxes']
#                 target['labels'] = target['labels'].long().to(device)
#             images = torch.stack([im.float().to(device) for im in ims], dim=0)
#             batch_losses, _ = model(images, targets)
#             loss = batch_losses['classification']
#             loss += batch_losses['bbox_regression']

#             ssd_classification_losses.append(batch_losses['classification'].item())
#             ssd_localization_losses.append(batch_losses['bbox_regression'].item())
#             loss = loss / acc_steps
#             loss.backward()

#             if (idx + 1) % acc_steps == 0:
#                 optimizer.step()
#                 optimizer.zero_grad()
#             if steps % train_config['log_steps'] == 0:
#                 loss_output = ''
#                 loss_output += 'SSD Classification Loss : {:.4f}'.format(np.mean(ssd_classification_losses))
#                 loss_output += ' | SSD Localization Loss : {:.4f}'.format(np.mean(ssd_localization_losses))
#                 print(loss_output)
#             if torch.isnan(loss):
#                 print('Loss is becoming nan. Exiting')
#                 raise RuntimeError('Loss is NaN')
#             steps += 1

#         if (idx + 1) % acc_steps != 0:
#             optimizer.step()
#             optimizer.zero_grad()

#         lr_scheduler.step()
#         print('Finished epoch {}'.format(i + 1))
#         loss_output = ''
#         loss_output += 'SSD Classification Loss : {:.4f}'.format(np.mean(ssd_classification_losses))
#         loss_output += ' | SSD Localization Loss : {:.4f}'.format(np.mean(ssd_localization_losses))
#         print(loss_output)

#         ckpt_dict = {
#             'epoch': i,
#             'steps': steps,
#             'model_state_dict': model.state_dict(),
#             'optimizer_state_dict': optimizer.state_dict(),
#             'lr_scheduler_state_dict': lr_scheduler.state_dict(),
#         }
#         torch.save(ckpt_dict, latest_ckpt_path)

#         if (i + 1) % 10 == 0:
#             numbered_path = os.path.join(train_config['task_name'],
#                                           f'epoch_{i+1}_' + train_config['ckpt_name'])
#             torch.save(ckpt_dict, numbered_path)
#             print(f'Saved periodic checkpoint at epoch {i+1}')

#     print('Done Training...')


# parser = argparse.ArgumentParser(description='Arguments for ssd training')
# parser.add_argument('--config', dest='config_path',
#                     default='config/voc.yaml', type=str)
# args = parser.parse_args(args=[])
# train(args)

In [ ]:
def load_model_and_dataset_for_infer(config_path='config/voc.yaml'):
    with open(config_path, 'r') as file:
        try:
            config = yaml.safe_load(file)
        except yaml.YAMLError as exc:
            print(exc)
    print(config)

    dataset_config = config['dataset_params']
    model_config = config['model_params']
    train_config = config['train_params']

    voc = VOCDataset('test', im_sets=dataset_config['test_im_sets'])
    test_dataset = DataLoader(voc, batch_size=1, shuffle=False)

    model = SSD(config=model_config, num_classes=dataset_config['num_classes'])
    model.to(device=device)
    model.eval()

    ckpt_path = os.path.join(train_config['task_name'], 'ssd_voc2007_weights.pth')
    assert os.path.exists(ckpt_path), "No checkpoint exists at {}".format(ckpt_path)

    checkpoint = torch.load(ckpt_path, map_location=device)
    if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
        model.load_state_dict(checkpoint['model_state_dict'])
    else:
        model.load_state_dict(checkpoint)

    return model, voc, test_dataset, config


def infer(config_path='config/voc.yaml', num_samples=5):
    if not os.path.exists('samples'):
        os.mkdir('samples')

    model, voc, test_dataset, config = load_model_and_dataset_for_infer(config_path)
    conf_threshold = config['train_params']['infer_conf_threshold']
    model.low_score_threshold = conf_threshold

    for i in tqdm(range(num_samples)):
        dataset_idx = random.randint(0, len(voc) - 1)
        im_tensor, target, fname = voc[dataset_idx]

        with torch.no_grad():
            _, ssd_detections = model(im_tensor.unsqueeze(0).to(device), [target])

        gt_im = cv2.imread(fname)
        h, w = gt_im.shape[:2]
        gt_im_copy = gt_im.copy()

        # Ground truth boxes
        for idx, box in enumerate(target['bboxes']):
            x1, y1, x2, y2 = box.detach().cpu().numpy()
            x1, y1, x2, y2 = int(w * x1), int(h * y1), int(w * x2), int(h * y2)
            cv2.rectangle(gt_im, (x1, y1), (x2, y2), thickness=2, color=[0, 255, 0])
            cv2.rectangle(gt_im_copy, (x1, y1), (x2, y2), thickness=2, color=[0, 255, 0])
            text = voc.idx2label[target['labels'][idx].detach().cpu().item()]
            text_size, _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_PLAIN, 1, 1)
            text_w, text_h = text_size
            cv2.rectangle(gt_im_copy, (x1, y1), (x1 + 10 + text_w, y1 + 10 + text_h), [255, 255, 255], -1)
            cv2.putText(gt_im, text=text, org=(x1 + 5, y1 + 15), thickness=1, fontScale=1,
                        color=[0, 0, 0], fontFace=cv2.FONT_HERSHEY_PLAIN)
            cv2.putText(gt_im_copy, text=text, org=(x1 + 5, y1 + 15), thickness=1, fontScale=1,
                        color=[0, 0, 0], fontFace=cv2.FONT_HERSHEY_PLAIN)
        cv2.addWeighted(gt_im_copy, 0.7, gt_im, 0.3, 0, gt_im)
        cv2.imwrite('samples/output_ssd_gt_{}.png'.format(i), gt_im)

        # Predicted boxes
        boxes = ssd_detections[0]['boxes']
        labels = ssd_detections[0]['labels']
        scores = ssd_detections[0]['scores']
        im = cv2.imread(fname)
        im_copy = im.copy()

        for idx, box in enumerate(boxes):
            x1, y1, x2, y2 = box.detach().cpu().numpy()
            x1, y1, x2, y2 = int(w * x1), int(h * y1), int(w * x2), int(h * y2)
            cv2.rectangle(im, (x1, y1), (x2, y2), thickness=2, color=[0, 0, 255])
            cv2.rectangle(im_copy, (x1, y1), (x2, y2), thickness=2, color=[0, 0, 255])
            text = '{} : {:.2f}'.format(voc.idx2label[labels[idx].detach().cpu().item()],
                                        scores[idx].detach().cpu().item())
            text_size, _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_PLAIN, 1, 1)
            text_w, text_h = text_size
            cv2.rectangle(im_copy, (x1, y1), (x1 + 10 + text_w, y1 + 10 + text_h), [255, 255, 255], -1)
            cv2.putText(im, text=text, org=(x1 + 5, y1 + 15), thickness=1, fontScale=1,
                        color=[0, 0, 0], fontFace=cv2.FONT_HERSHEY_PLAIN)
            cv2.putText(im_copy, text=text, org=(x1 + 5, y1 + 15), thickness=1, fontScale=1,
                        color=[0, 0, 0], fontFace=cv2.FONT_HERSHEY_PLAIN)
        cv2.addWeighted(im_copy, 0.7, im, 0.3, 0, im)
        cv2.imwrite('samples/output_ssd_{}.jpg'.format(i), im)

        gt_im_rgb = cv2.cvtColor(gt_im, cv2.COLOR_BGR2RGB)
        im_rgb = cv2.cvtColor(im, cv2.COLOR_BGR2RGB)

        fig, axes = plt.subplots(1, 2, figsize=(14, 7))
        axes[0].imshow(gt_im_rgb)
        axes[0].set_title('Ground Truth')
        axes[0].axis('off')
        axes[1].imshow(im_rgb)
        axes[1].set_title('Prediction')
        axes[1].axis('off')
        plt.tight_layout()
        plt.show()

    print('Done Detecting...')


infer(config_path='config/voc.yaml', num_samples=5)